<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/03_watermark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Watermark channel stress test

Manuscript section 3.6.1 and research question 1: does an embedded watermark survive social-media processing?

Each AI test image gets an invisible keyed pattern added, `I_w = I + αW` (as a generator would do), human images stay unmarked.
Every image then goes through the same transforms as the CNN stress test (JPEG 90/70/50/30, WebP, downscaling, recompression),
and a correlation detector decides whether the mark is still there. Three strengths α are compared (stronger = more durable but more visible).

Before starting: notebook 01 has saved `thesis-data/harmonized.zip` to Drive. **No GPU needed**: Runtime > Change runtime type > **CPU**.
Takes about 10-20 minutes.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e ".[train,dev]"

## 2. Restore the prepared data from Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
ZIP = "/content/drive/MyDrive/thesis-data/harmonized.zip"
if not os.path.exists(ZIP):
    raise FileNotFoundError(f"{ZIP} not found: run notebook 01 through step 8 first, it saves the prepared data there")
!unzip -qo $ZIP -d .
!wc -l data/harmonized/val.csv data/harmonized/test.csv

## 3. Run the watermark stress test
First sets the detection threshold on the validation images so that at most 1% of *unmarked* images would be flagged
under any transform (falsely flagging human art is the costly mistake). Then scores every test image under every transform.
Results are written to Google Drive (`thesis-data/watermark/`).

In [ ]:
!mkdir -p /content/drive/MyDrive/thesis-data/watermark && rm -rf outputs/watermark && ln -s /content/drive/MyDrive/thesis-data/watermark outputs/watermark
!python scripts/watermark_test.py

## 4. Results
**Detection rate**: share of watermarked AI images where the mark is still found (watermark survival).
**False detection rate**: share of unmarked human images wrongly flagged. **PSNR**: how invisible the mark is (above ~40 dB is generally invisible, ~30 dB starts to show on flat areas).

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("outputs/watermark/results.csv")
order = df.condition.unique()
print("threshold:", json.load(open("outputs/watermark/threshold.json"))["threshold"])
print("PSNR (dB) per alpha:", df.groupby("alpha").psnr_db.first().round(1).to_dict())
print("\nDetection rate (watermark survival):")
display(df.pivot(index="condition", columns="alpha", values="detection_rate").loc[order].round(4))
print("False detection rate on unmarked human images:")
display(df.pivot(index="condition", columns="alpha", values="false_detection_rate").loc[order].round(4))
ax = df.pivot(index="condition", columns="alpha", values="detection_rate").loc[order].plot.bar(figsize=(10, 3.5), ylim=(0, 1.05))
ax.set_ylabel("detection rate"); ax.set_title("Watermark survival under social-media transforms"); ax.legend(title="alpha")
plt.tight_layout()

## 5. What the watermark looks like
Original AI image, watermarked at the default α, and the difference amplified 20× so the pattern is visible.

In [ ]:
import numpy as np
from PIL import Image
from attribution.config import load_config
from attribution.data.splits import read_split
from attribution.signals import watermark
cfg = load_config()
r = next(r for r in read_split("data/harmonized/test.csv") if r.label == 1)
img = Image.open(r.path).convert("RGB")
marked = watermark.embed(img, cfg["watermark"]["alpha"], cfg["watermark"]["key"])
diff = (np.asarray(marked, float) - np.asarray(img, float)) * 20 + 128
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in zip(axes, [img, marked, Image.fromarray(diff.clip(0, 255).astype(np.uint8))],
                    ["original", f"watermarked (alpha={cfg['watermark']['alpha']}, PSNR {watermark.psnr(img, marked):.1f} dB)", "difference x20"]):
    a.imshow(im); a.set_title(t); a.axis("off")
plt.tight_layout()